In [1]:
!pip install -q "transformers>=4.51.0" "datasets>=2.19.0" "accelerate>=0.30.0" "lm-eval>=0.4.8" pyyaml matplotlib

import os
REPO_URL = "https://github.com/syedmoazzamrazagilani/ATML-PA3.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

%cd {REPO_DIR}
!wget -q -O first_plot_questions.yaml https://raw.githubusercontent.com/emergent-misalignment/emergent-misalignment/main/evaluation/first_plot_questions.yaml
!rm -rf /kaggle/working/task1_outputs /kaggle/working/task1_smoke
!mkdir -p /kaggle/working/task1_outputs

  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


Cloning into '/kaggle/working/repo'...


remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 87 (delta 43), reused 0 (delta 0), pack-reused 0 (from 0)


Receiving objects: 100% (87/87), 49.37 KiB | 8.23 MiB/s, done.
Resolving deltas: 100% (43/43), done.


/kaggle/working/repo


In [2]:
import subprocess, shutil

subprocess.run([
    "python", "-m", "task1.run_task1",
    "--stage", "smoke",
    "--variant", "all",
    "--output-dir", "/kaggle/working/task1_smoke",
    "--yaml-path", "first_plot_questions.yaml",
], check=True)

shutil.rmtree("/kaggle/working/task1_smoke", ignore_errors=True)
print("\nSmoke test passed 100% cleanly! Starting full Task 1 pipeline.\n")

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 291664.22 examples/s]


[Data Prep] Retained 64 examples. Stats: {'inspected': 64, 'excluded_empty': 0, 'excluded_gsm8k_overlap': 0, 'excluded_over_length': 0, 'retained': 64}

--- Training Variant: full_ft (2 updates) ---


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:  61%|██████▏   | 191/311 [00:00<00:00, 446.78it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 412.35it/s]



--- Training Variant: l2_sp (2 updates) ---


... (123 outputs left out)

Running loglikelihood requests:  31%|███▏      | 25/80 [00:00<00:00, 62.37it/s]

Running loglikelihood requests:  72%|███████▎  | 58/80 [00:00<00:00, 101.56it/s]

Running loglikelihood requests: 100%|██████████| 80/80 [00:00<00:00, 92.99it/s] 



Smoke test passed 100% cleanly! Starting full Task 1 pipeline.



In [3]:
import subprocess

subprocess.run([
    "python", "-m", "task1.run_task1",
    "--stage", "prep",
    "--output-dir", "/kaggle/working/task1_outputs",
    "--yaml-path", "first_plot_questions.yaml",
], check=True)

Generating train split:   0%|          | 0/395000 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 395000/395000 [00:06<00:00, 60317.77 examples/s]


[Data Prep] Retained 40000 examples. Stats: {'inspected': 40002, 'excluded_empty': 0, 'excluded_gsm8k_overlap': 0, 'excluded_over_length': 2, 'retained': 40000}


CompletedProcess(args=['python', '-m', 'task1.run_task1', '--stage', 'prep', '--output-dir', '/kaggle/working/task1_outputs', '--yaml-path', 'first_plot_questions.yaml'], returncode=0)

In [4]:
import subprocess

subprocess.run([
    "python", "-m", "task1.run_task1",
    "--stage", "train",
    "--variant", "all",
    "--output-dir", "/kaggle/working/task1_outputs",
    "--prune-optimizer-states",
], check=True)


--- Training Variant: full_ft (1250 updates) ---


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:  51%|█████     | 158/311 [00:00<00:00, 464.99it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 443.03it/s]



--- Training Variant: l2_sp (1250 updates) ---


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (7 outputs left out)


--- Training Variant: lora_16 (1250 updates) ---


Loading weights:  73%|███████▎  | 226/311 [00:00<00:00, 559.94it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 482.14it/s]



--- Training Variant: oplora_16 (1250 updates) ---


In [ ]:
import subprocess

subprocess.run([
    "python", "-m", "task1.run_task1",
    "--stage", "eval_and_diag",
    "--output-dir", "/kaggle/working/task1_outputs",
    "--yaml-path", "first_plot_questions.yaml",
], check=True)

In [ ]:
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display

out_dir = Path("/kaggle/working/task1_outputs")
bundle_dir = Path("/kaggle/working/task1_report_bundle")
shutil.rmtree(bundle_dir, ignore_errors=True)
bundle_dir.mkdir(parents=True, exist_ok=True)

for sub in ["eval", "figures"]:
    if (out_dir / sub).exists():
        shutil.copytree(out_dir / sub, bundle_dir / sub, dirs_exist_ok=True)

for fname in ["data_selection_stats.json", "probe_ids.json"]:
    if (out_dir / fname).exists():
        shutil.copy2(out_dir / fname, bundle_dir / fname)

for var in ["full_ft", "l2_sp", "lora_4", "lora_16", "oplora_16"]:
    var_src = out_dir / var
    var_dst = bundle_dir / var
    if var_src.exists():
        var_dst.mkdir(parents=True, exist_ok=True)
        for log_name in ["train_log.jsonl", "run_summary.json", "merged_logit_verification.json"]:
            if (var_src / log_name).exists():
                shutil.copy2(var_src / log_name, var_dst / log_name)
        ckpt_src = var_src / "checkpoints"
        if ckpt_src.exists():
            for pt_file in ckpt_src.glob("adapter_update_*.pt"):
                (var_dst / "checkpoints").mkdir(parents=True, exist_ok=True)
                shutil.copy2(pt_file, var_dst / "checkpoints" / pt_file.name)

os.chdir("/kaggle/working")
zip_base = "/kaggle/working/task1_report_artifacts"
shutil.make_archive(zip_base, "zip", bundle_dir)
shutil.rmtree(bundle_dir, ignore_errors=True)

zip_file = Path("task1_report_artifacts.zip")
size_mb = zip_file.stat().st_size / (1024 ** 2)
print("\n" + "=" * 60)
print(f"Task 1 artifacts packaged: {zip_file.name} ({size_mb:.2f} MB)")
print("=" * 60 + "\n")
display(FileLink("task1_report_artifacts.zip"))